# BTL02 – TV4: Trực quan hóa Filter, E4 Pretrained, Đa seed & Phân tích

**Thành viên:** Lê Thị Thanh Tuyền – 3123411332  
**Vai trò:** Trực quan hóa filter, fine-tune pretrained (E4), chạy mô hình tốt nhất với nhiều seed, phân tích & tổng hợp  
**Môn:** Học sâu – DCT123C4 – Nhóm 07-09  
**Ngày chạy:** 24/11/2026

---

> **Yêu cầu Kaggle:**
> - **GPU T4** (hoặc GPU khả dụng khác) – bật trong Settings → Accelerator
> - **Internet ON** – để tải pretrained weights từ torchvision  
> - **FER2013 CSV** – Add Input dataset chứa `fer2013.csv`

## Bước 0: Kiểm tra môi trường GPU & Clone repo

In [ ]:
!nvidia-smi
from datetime import datetime, timezone, timedelta
LOCAL_TZ = timezone(timedelta(hours=7), name="UTC+07")
print("Local time (UTC+7):", datetime.now(LOCAL_TZ).isoformat(timespec="seconds"))
print("Note: nvidia-smi above shows the Kaggle host clock in UTC.")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!git clone https://github.com/Truongbuiii/SGU26K2_HocSau_S5_DCT123C4_0709_Nhom.git /kaggle/working/repo
%cd /kaggle/working/repo/Tuan11

## Bước 1: Xác định đường dẫn FER2013 CSV & Kiểm tra dữ liệu

In [ ]:
import os
import glob

# Tìm file fer2013.csv trong /kaggle/input
found = glob.glob('/kaggle/input/**/fer2013.csv', recursive=True)
print("Các file fer2013.csv tìm thấy:")
for f in found:
    print(f"  {f}  ({os.path.getsize(f) / 1e6:.1f} MB)")

# Chọn file đầu tiên tìm được
CSV = found[0] if found else None
print(f"\n=> Sử dụng: {CSV}")
assert CSV is not None, "Chưa tìm thấy fer2013.csv! Hãy Add Input dataset."

# Đặt biến môi trường để các script src/ sử dụng
os.environ['FER2013_CSV'] = CSV

In [ ]:
# Kiểm tra dữ liệu: phải đạt 28709 / 3589 / 3589
!python src/dataset.py

## Bước 2: Cài đặt thư viện còn thiếu (nếu cần)

In [ ]:
# Cài scikit-learn nếu chưa có (cần cho evaluate)
try:
    import sklearn
    print(f"scikit-learn đã có: {sklearn.__version__}")
except ImportError:
    !pip install -q scikit-learn
    import sklearn
    print(f"Đã cài scikit-learn: {sklearn.__version__}")

try:
    import matplotlib
    print(f"matplotlib đã có: {matplotlib.__version__}")
except ImportError:
    !pip install -q matplotlib

---
## Phần A: E4-01 – Fine-tune ResNet18 (ImageNet pretrained)

Fine-tune ResNet18 pretrained trên ImageNet. Pipeline:
- FER2013 grayscale 48×48 → replicate 3 kênh RGB → resize 96×96 → normalize ImageNet mean/std
- Thay lớp FC cuối → 7 lớp cảm xúc
- AdamW, lr=1e-4, 30 epochs, patience=7, AMP trên GPU

In [ ]:
!python src/train_pretrained.py \
    --arch resnet18 \
    --seed 42 \
    --csv_path "$FER2013_CSV" \
    --output_root /kaggle/working/results/runs

In [ ]:
# Kiểm tra kết quả E4-01
import json
from pathlib import Path

run_e4_resnet = Path('/kaggle/working/results/runs/E4-resnet18_resnet18_s42')
if (run_e4_resnet / 'best_metrics.json').exists():
    metrics = json.loads((run_e4_resnet / 'best_metrics.json').read_text())
    print(f"E4-01 ResNet18 - Best epoch: {metrics['epoch']}, Val loss: {metrics['val_loss']:.4f}, Val acc: {metrics['val_acc']:.4f}")
else:
    print("Chưa có best_metrics.json!")

# Kiểm tra best.pt tồn tại
print(f"best.pt exists: {(run_e4_resnet / 'best.pt').exists()}")
print(f"Files: {[f.name for f in run_e4_resnet.iterdir()]}")

## Phần B: E4-02 – Fine-tune VGG16 (ImageNet pretrained)

Cùng pipeline như ResNet18, nhưng VGG16 nặng hơn nhiều (138M params). Thời gian chạy lâu hơn.

In [ ]:
!python src/train_pretrained.py \
    --arch vgg16 \
    --seed 42 \
    --csv_path "$FER2013_CSV" \
    --output_root /kaggle/working/results/runs

In [ ]:
# Kiểm tra kết quả E4-02
run_e4_vgg = Path('/kaggle/working/results/runs/E4-vgg16_vgg16_s42')
if (run_e4_vgg / 'best_metrics.json').exists():
    metrics = json.loads((run_e4_vgg / 'best_metrics.json').read_text())
    print(f"E4-02 VGG16 - Best epoch: {metrics['epoch']}, Val loss: {metrics['val_loss']:.4f}, Val acc: {metrics['val_acc']:.4f}")
else:
    print("Chưa có best_metrics.json!")

print(f"best.pt exists: {(run_e4_vgg / 'best.pt').exists()}")

## So sánh E4-01 (ResNet18) vs E4-02 (VGG16)

In [ ]:
import pandas as pd

rows = []
for label, run_dir in [('E4-01 ResNet18', run_e4_resnet), ('E4-02 VGG16', run_e4_vgg)]:
    bm = run_dir / 'best_metrics.json'
    env = run_dir / 'env.json'
    if bm.exists():
        m = json.loads(bm.read_text())
        e = json.loads(env.read_text()) if env.exists() else {}
        rows.append({
            'Run': label,
            'Best Epoch': m['epoch'],
            'Val Loss': f"{m['val_loss']:.4f}",
            'Val Acc (%)': f"{m['val_acc']*100:.2f}" if m['val_acc'] < 1 else f"{m['val_acc']:.2f}",
            'Time (s)': f"{e.get('seconds', 'N/A')}",
            'GPU': e.get('gpu', 'N/A')
        })

df_e4 = pd.DataFrame(rows)
print("\n=== So sánh E4 Transfer Learning ===")
print(df_e4.to_string(index=False))

---
## Phần C: E4-03 – Chạy cấu hình CNN tốt nhất với 3 seed (42, 43, 44)

Theo `results/summary.csv`, cấu hình tốt nhất từ E1–E3 là **E2-04**:  
Medium, ReLU, không BN, conv_dropout=0.2, fc_dropout=0.5, `flip_affine`, `div255`, Adam lr=1e-4, batch 64, 100 epochs, patience 10.

Chạy 3 seed để đo độ ổn định (mean ± std).

In [ ]:
# Seed 42
!FER2013_CSV="$FER2013_CSV" python src/train.py \
    --exp_id E4-03 \
    --exp_name best_seed42 \
    --runner TV4 \
    --arch medium \
    --activation relu \
    --conv_dropout 0.2 \
    --fc_dropout 0.5 \
    --optimizer adam \
    --lr 1e-4 \
    --batch_size 64 \
    --aug_name flip_affine \
    --norm div255 \
    --epochs 100 \
    --patience 10 \
    --seed 42

In [ ]:
# Seed 43
!FER2013_CSV="$FER2013_CSV" python src/train.py \
    --exp_id E4-03 \
    --exp_name best_seed43 \
    --runner TV4 \
    --arch medium \
    --activation relu \
    --conv_dropout 0.2 \
    --fc_dropout 0.5 \
    --optimizer adam \
    --lr 1e-4 \
    --batch_size 64 \
    --aug_name flip_affine \
    --norm div255 \
    --epochs 100 \
    --patience 10 \
    --seed 43

In [ ]:
# Seed 44
!FER2013_CSV="$FER2013_CSV" python src/train.py \
    --exp_id E4-03 \
    --exp_name best_seed44 \
    --runner TV4 \
    --arch medium \
    --activation relu \
    --conv_dropout 0.2 \
    --fc_dropout 0.5 \
    --optimizer adam \
    --lr 1e-4 \
    --batch_size 64 \
    --aug_name flip_affine \
    --norm div255 \
    --epochs 100 \
    --patience 10 \
    --seed 44

### Tổng hợp kết quả 3 seed (Mean ± Std)

In [ ]:
# Tổng hợp bằng summarize_tv4.py
!python src/summarize_tv4.py \
    --run_dirs results/runs/E4-03_best_seed42_s42 results/runs/E4-03_best_seed43_s43 results/runs/E4-03_best_seed44_s44 \
    --out results/summary_TV4_seeds.csv

In [ ]:
# Hiển thị bảng tổng hợp
import pandas as pd

summary_path = Path('results/summary_TV4_seeds.csv')
if summary_path.exists():
    df_seeds = pd.read_csv(summary_path)
    print("\n=== Kết quả 3 seed – Mô hình tốt nhất (E2-04 config) ===")
    print(df_seeds.to_string(index=False))
else:
    print("Chưa có file summary!")

---
## Phần D: Trực quan hóa Filter & Feature Map

Dùng checkpoint CNN tốt nhất (seed 42) để:
1. Vẽ kernel lớp convolution đầu tiên
2. Vẽ feature map lớp convolution cuối (theo từng lớp cảm xúc)
3. Activation maximization cho một số filter cuối

In [ ]:
!python src/visualize_filters.py \
    --run_dir results/runs/E4-03_best_seed42_s42 \
    --csv_path "$FER2013_CSV" \
    --out_dir results/figures

In [ ]:
# Hiển thị các hình đã tạo
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir = Path('results/figures')
tag = 'E4-03_best_seed42_s42'

for suffix in ['_first_kernels.png', '_last_feature_maps.png', '_activation_maximization.png']:
    fpath = fig_dir / f"{tag}{suffix}"
    if fpath.exists():
        print(f"\n=== {fpath.name} ===")
        img = mpimg.imread(str(fpath))
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(fpath.name)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Chưa có: {fpath}")

## Phần E: Grad-CAM (tùy chọn)

In [ ]:
!python src/gradcam.py \
    --run_dir results/runs/E4-03_best_seed42_s42 \
    --csv_path "$FER2013_CSV" \
    --out results/figures/gradcam_final.png

In [ ]:
# Hiển thị Grad-CAM
gradcam_path = Path('results/figures/gradcam_final.png')
if gradcam_path.exists():
    img = mpimg.imread(str(gradcam_path))
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Grad-CAM on PublicTest')
    plt.tight_layout()
    plt.show()
else:
    print(f"Chưa có: {gradcam_path}")

---
## Phần F: Đánh giá PrivateTest (chỉ chạy sau khi khóa mô hình)

> **QUAN TRỌNG:** Chỉ đánh giá trên PrivateTest SAU KHI đã chốt cấu hình và checkpoint theo PublicTest (validation). KHÔNG dùng kết quả PrivateTest để đổi lựa chọn mô hình.

### F1: Đánh giá CNN tốt nhất (E4-03, seed 42)

In [ ]:
!python src/evaluate_final_cnn.py \
    --run_dir results/runs/E4-03_best_seed42_s42 \
    --csv_path "$FER2013_CSV"

In [ ]:
# Hiển thị confusion matrix PrivateTest – CNN
cnn_cm_path = Path('results/runs/E4-03_best_seed42_s42/private_test_confusion_matrix.png')
if cnn_cm_path.exists():
    img = mpimg.imread(str(cnn_cm_path))
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('CNN Final - PrivateTest Confusion Matrix')
    plt.tight_layout()
    plt.show()

# Hiển thị per-class metrics
pt_metrics_path = Path('results/runs/E4-03_best_seed42_s42/private_test_metrics.json')
if pt_metrics_path.exists():
    pt_metrics = json.loads(pt_metrics_path.read_text())
    print(f"\nPrivateTest Accuracy: {pt_metrics['accuracy']:.4f} ({pt_metrics['accuracy']*100:.2f}%)")
    cr = pt_metrics['classification_report']
    print(f"\n{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print('-' * 55)
    for cls in ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']:
        if cls in cr:
            c = cr[cls]
            print(f"{cls:<12} {c['precision']:>10.4f} {c['recall']:>10.4f} {c['f1-score']:>10.4f} {c['support']:>10}")

### F2: Đánh giá ResNet18 pretrained (E4-01)

In [ ]:
!python src/evaluate_pretrained.py \
    --run_dir /kaggle/working/results/runs/E4-resnet18_resnet18_s42 \
    --csv_path "$FER2013_CSV"

In [ ]:
# Hiển thị kết quả PrivateTest – ResNet18
resnet_pt_path = Path('/kaggle/working/results/runs/E4-resnet18_resnet18_s42/private_test_metrics.json')
if resnet_pt_path.exists():
    resnet_pt = json.loads(resnet_pt_path.read_text())
    print(f"ResNet18 PrivateTest Accuracy: {resnet_pt['accuracy']:.4f} ({resnet_pt['accuracy']*100:.2f}%)")

    # Confusion matrix
    cm_path = resnet_pt_path.parent / 'private_test_confusion_matrix.png'
    if cm_path.exists():
        img = mpimg.imread(str(cm_path))
        fig, ax = plt.subplots(figsize=(9, 8))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title('ResNet18 Pretrained - PrivateTest Confusion Matrix')
        plt.tight_layout()
        plt.show()

### F3: Đánh giá VGG16 pretrained (E4-02)

In [ ]:
!python src/evaluate_pretrained.py \
    --run_dir /kaggle/working/results/runs/E4-vgg16_vgg16_s42 \
    --csv_path "$FER2013_CSV"

In [ ]:
# Hiển thị kết quả PrivateTest – VGG16
vgg_pt_path = Path('/kaggle/working/results/runs/E4-vgg16_vgg16_s42/private_test_metrics.json')
if vgg_pt_path.exists():
    vgg_pt = json.loads(vgg_pt_path.read_text())
    print(f"VGG16 PrivateTest Accuracy: {vgg_pt['accuracy']:.4f} ({vgg_pt['accuracy']*100:.2f}%)")

    cm_path = vgg_pt_path.parent / 'private_test_confusion_matrix.png'
    if cm_path.exists():
        img = mpimg.imread(str(cm_path))
        fig, ax = plt.subplots(figsize=(9, 8))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title('VGG16 Pretrained - PrivateTest Confusion Matrix')
        plt.tight_layout()
        plt.show()

### F4: Đánh giá PrivateTest cho seed 43 và 44

In [ ]:
# Seed 43
!python src/evaluate_final_cnn.py \
    --run_dir results/runs/E4-03_best_seed43_s43 \
    --csv_path "$FER2013_CSV"

# Seed 44
!python src/evaluate_final_cnn.py \
    --run_dir results/runs/E4-03_best_seed44_s44 \
    --csv_path "$FER2013_CSV"

---
## Phần G: Tổng hợp toàn bộ kết quả TV4

In [ ]:
import json, statistics
from pathlib import Path
import pandas as pd

print("=" * 80)
print("           TỔNG HỢP KẾT QUẢ TV4 – LÊ THỊ THANH TUYỀN – 3123411332")
print("=" * 80)

# --- E4-01 / E4-02: Pretrained ---
print("\n--- E4: Transfer Learning (Pretrained) ---")
for label, run_dir in [('E4-01 ResNet18', Path('/kaggle/working/results/runs/E4-resnet18_resnet18_s42')),
                       ('E4-02 VGG16', Path('/kaggle/working/results/runs/E4-vgg16_vgg16_s42'))]:
    bm = run_dir / 'best_metrics.json'
    pt = run_dir / 'private_test_metrics.json'
    env = run_dir / 'env.json'
    if bm.exists():
        m = json.loads(bm.read_text())
        e = json.loads(env.read_text()) if env.exists() else {}
        p = json.loads(pt.read_text()) if pt.exists() else {}
        va_str = f"{m['val_acc']*100:.2f}%" if m['val_acc'] < 1 else f"{m['val_acc']:.2f}%"
        pt_str = f"{p['accuracy']*100:.2f}%" if p.get('accuracy') else 'N/A'
        print(f"  {label}: best_epoch={m['epoch']}, val_acc={va_str}, val_loss={m['val_loss']:.4f}, "
              f"test_acc={pt_str}, time={e.get('seconds','N/A')}s")

# --- E4-03: 3 seeds ---
print("\n--- E4-03: CNN tốt nhất qua 3 seed ---")
seed_results = []
for seed in [42, 43, 44]:
    run = Path(f'results/runs/E4-03_best_seed{seed}_s{seed}')
    hist_path = run / 'history.csv'
    pt_path = run / 'private_test_metrics.json'
    if hist_path.exists():
        import csv
        with hist_path.open(newline='', encoding='utf-8-sig') as f:
            rows = list(csv.DictReader(f))
        best = min(rows, key=lambda r: float(r['val_loss']))
        pt_acc = None
        if pt_path.exists():
            pt_data = json.loads(pt_path.read_text())
            pt_acc = pt_data['accuracy']
        result = {
            'seed': seed,
            'best_epoch': int(float(best['epoch'])),
            'val_acc': float(best['val_acc']),
            'val_loss': float(best['val_loss']),
            'test_acc': pt_acc
        }
        seed_results.append(result)
        pt_str = f"{pt_acc*100:.2f}%" if pt_acc else 'N/A'
        va_str = f"{result['val_acc']*100:.2f}%" if result['val_acc'] < 1 else f"{result['val_acc']:.2f}%"
        print(f"  Seed {seed}: best_epoch={result['best_epoch']}, val_acc={va_str}, "
              f"val_loss={result['val_loss']:.4f}, test_acc={pt_str}")

if len(seed_results) >= 2:
    val_accs = [r['val_acc'] for r in seed_results]
    val_losses = [r['val_loss'] for r in seed_results]
    test_accs = [r['test_acc'] for r in seed_results if r['test_acc'] is not None]
    
    mean_va = statistics.mean(val_accs)
    std_va = statistics.stdev(val_accs) if len(val_accs) >= 2 else 0
    mean_vl = statistics.mean(val_losses)
    std_vl = statistics.stdev(val_losses) if len(val_losses) >= 2 else 0
    
    if mean_va < 1:  # fraction
        print(f"\n  Mean ± Std Val Acc:  {mean_va*100:.2f}% ± {std_va*100:.2f}%")
    else:
        print(f"\n  Mean ± Std Val Acc:  {mean_va:.2f}% ± {std_va:.2f}%")
    print(f"  Mean ± Std Val Loss: {mean_vl:.4f} ± {std_vl:.4f}")
    if test_accs:
        mean_ta = statistics.mean(test_accs)
        std_ta = statistics.stdev(test_accs) if len(test_accs) >= 2 else 0
        print(f"  Mean ± Std Test Acc: {mean_ta*100:.2f}% ± {std_ta*100:.2f}%")

print("\n" + "=" * 80)

---
## Phần H: Cập nhật nhật ký thí nghiệm TV4

In [ ]:
import csv, json, datetime
from pathlib import Path

log_rows = []
LOCAL_TZ = datetime.timezone(datetime.timedelta(hours=7), name='UTC+07')
def local_run_start(env):
    """Convert run start_time to UTC+7, including legacy UTC timestamps."""
    value = env.get('start_time')
    if not value:
        return ''
    return datetime.datetime.fromisoformat(value).astimezone(LOCAL_TZ).strftime('%Y-%m-%d %H:%M')

# E4-01 ResNet18
run = Path('/kaggle/working/results/runs/E4-resnet18_resnet18_s42')
bm = json.loads((run / 'best_metrics.json').read_text()) if (run / 'best_metrics.json').exists() else {}
env = json.loads((run / 'env.json').read_text()) if (run / 'env.json').exists() else {}
pt = json.loads((run / 'private_test_metrics.json').read_text()) if (run / 'private_test_metrics.json').exists() else {}
log_rows.append({
    'ExpID': 'E4-01', 'date_time': local_run_start(env), 'runner': 'TV4',
    'purpose': 'ResNet18 ImageNet pretrained fine-tune',
    'change_vs_baseline': 'ResNet18 ImageNet pretrained; resize 96; RGB replicate; ImageNet normalization',
    'seed': 42, 'best_epoch': bm.get('epoch',''), 'val_loss': f"{bm.get('val_loss',0):.4f}" if bm else '',
    'val_acc': f"{bm.get('val_acc',0):.4f}" if bm else '', 'test_acc': f"{pt.get('accuracy',0):.4f}" if pt else '',
    'runtime_minutes': f"{env.get('seconds',0)/60:.1f}" if env.get('seconds') else '',
    'run_dir': str(run), 'status': 'DONE' if bm else 'ERROR', 'notes': ''
})

# E4-02 VGG16
run = Path('/kaggle/working/results/runs/E4-vgg16_vgg16_s42')
bm = json.loads((run / 'best_metrics.json').read_text()) if (run / 'best_metrics.json').exists() else {}
env = json.loads((run / 'env.json').read_text()) if (run / 'env.json').exists() else {}
pt = json.loads((run / 'private_test_metrics.json').read_text()) if (run / 'private_test_metrics.json').exists() else {}
log_rows.append({
    'ExpID': 'E4-02', 'date_time': local_run_start(env), 'runner': 'TV4',
    'purpose': 'VGG16 ImageNet pretrained fine-tune',
    'change_vs_baseline': 'VGG16 ImageNet pretrained; resize 96; RGB replicate; ImageNet normalization',
    'seed': 42, 'best_epoch': bm.get('epoch',''), 'val_loss': f"{bm.get('val_loss',0):.4f}" if bm else '',
    'val_acc': f"{bm.get('val_acc',0):.4f}" if bm else '', 'test_acc': f"{pt.get('accuracy',0):.4f}" if pt else '',
    'runtime_minutes': f"{env.get('seconds',0)/60:.1f}" if env.get('seconds') else '',
    'run_dir': str(run), 'status': 'DONE' if bm else 'ERROR', 'notes': ''
})

# E4-03 seeds
for seed in [42, 43, 44]:
    run = Path(f'results/runs/E4-03_best_seed{seed}_s{seed}')
    hist_path = run / 'history.csv'
    pt_path = run / 'private_test_metrics.json'
    env_path = run / 'env.json'
    best_epoch = val_loss = val_acc = test_acc = runtime = ''
    status = 'NOT_RUN'
    if hist_path.exists():
        with hist_path.open(newline='', encoding='utf-8-sig') as f:
            rows = list(csv.DictReader(f))
        if rows:
            best = min(rows, key=lambda r: float(r['val_loss']))
            best_epoch = int(float(best['epoch']))
            val_loss = f"{float(best['val_loss']):.4f}"
            val_acc = f"{float(best['val_acc']):.4f}"
            status = 'DONE'
    if pt_path.exists():
        pt_data = json.loads(pt_path.read_text())
        test_acc = f"{pt_data['accuracy']:.4f}"
    run_date = ''
    if env_path.exists():
        env_data = json.loads(env_path.read_text())
        run_date = local_run_start(env_data)
        if 'seconds' in env_data:
            runtime = f"{env_data['seconds']/60:.1f}"
        elif 'end_time' in env_data and 'start_time' in env_data:
            runtime = 'see env.json'
    log_rows.append({
        'ExpID': 'E4-03', 'date_time': run_date, 'runner': 'TV4',
        'purpose': f'Final selected model seed {seed}',
        'change_vs_baseline': 'Final selected model; same config across seeds (E2-04: Adam lr=1e-4)',
        'seed': seed, 'best_epoch': best_epoch, 'val_loss': val_loss,
        'val_acc': val_acc, 'test_acc': test_acc,
        'runtime_minutes': runtime, 'run_dir': str(run),
        'status': status, 'notes': ''
    })

# Ghi file nhật ký
log_path = Path('results/experiment_log_TV4.csv')
fieldnames = ['ExpID','date_time','runner','purpose','change_vs_baseline','seed','best_epoch','val_loss','val_acc','test_acc','runtime_minutes','run_dir','status','notes']
with log_path.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(log_rows)

print(f"Đã cập nhật nhật ký: {log_path}")
print(pd.read_csv(log_path).to_string(index=False))

---
## Phần I: Learning curves cho các run TV4

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Vẽ learning curves cho 3 seeds
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, seed in enumerate([42, 43, 44]):
    hist_path = Path(f'results/runs/E4-03_best_seed{seed}_s{seed}/history.csv')
    if hist_path.exists():
        df = pd.read_csv(hist_path)
        ax = axes[idx]
        ax.plot(df['epoch'], df['train_acc'], label='Train Acc', color='blue')
        ax.plot(df['epoch'], df['val_acc'], label='Val Acc', color='orange')
        best_epoch = df.loc[df['val_loss'].idxmin(), 'epoch']
        ax.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best epoch={int(best_epoch)}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.set_title(f'Seed {seed}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

fig.suptitle('E4-03: Learning Curves – 3 Seeds (cấu hình E2-04)', fontsize=13)
plt.tight_layout()
plt.savefig('results/figures/E4-03_learning_curves_3seeds.png', dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: results/figures/E4-03_learning_curves_3seeds.png")

In [ ]:
# Vẽ learning curves cho E4-01 (ResNet18) và E4-02 (VGG16)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for idx, (label, run_dir) in enumerate([('E4-01 ResNet18', Path('/kaggle/working/results/runs/E4-resnet18_resnet18_s42')),
                                         ('E4-02 VGG16', Path('/kaggle/working/results/runs/E4-vgg16_vgg16_s42'))]):
    hist_path = run_dir / 'history.csv'
    if hist_path.exists():
        df = pd.read_csv(hist_path)
        ax = axes[idx]
        ax.plot(df['epoch'], df['train_acc'], label='Train Acc', color='blue')
        ax.plot(df['epoch'], df['val_acc'], label='Val Acc', color='orange')
        best_epoch = df.loc[df['val_loss'].idxmin(), 'epoch']
        ax.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best epoch={int(best_epoch)}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.set_title(label)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

fig.suptitle('E4: Pretrained Transfer Learning – Learning Curves', fontsize=13)
plt.tight_layout()
plt.savefig('results/figures/E4_pretrained_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: results/figures/E4_pretrained_learning_curves.png")

---
## Phần J: Copy kết quả về repo để commit

In [ ]:
import shutil
from pathlib import Path

# Copy các run E4 pretrained từ /kaggle/working vào results/runs/ của repo
for run_name in ['E4-resnet18_resnet18_s42', 'E4-vgg16_vgg16_s42']:
    src = Path(f'/kaggle/working/results/runs/{run_name}')
    dst = Path(f'results/runs/{run_name}')
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            if f.name != 'best.pt':  # Không copy checkpoint lớn vào git
                shutil.copy2(f, dst / f.name)
                print(f"  Copied: {run_name}/{f.name}")

print("\n=== Liệt kê tất cả run dirs ===")
runs_dir = Path('results/runs')
for d in sorted(runs_dir.iterdir()):
    if d.is_dir() and d.name.startswith('E4'):
        files = [f.name for f in d.iterdir()]
        print(f"  {d.name}/: {files}")

## Phần K: Liệt kê toàn bộ output để lưu minh chứng

Dùng **Save Version → Save & Run All** trên Kaggle để lưu toàn bộ notebook output.  
Chụp screenshots các ô kết quả quan trọng để lưu vào `evidence/ThanhVien4/`.

In [ ]:
import os
from pathlib import Path

print("=== TOÀN BỘ FILE KẾT QUẢ ĐÃ TẠO ===")
print("\n--- results/figures/ ---")
fig_dir = Path('results/figures')
if fig_dir.exists():
    for f in sorted(fig_dir.iterdir()):
        if f.is_file():
            print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

print("\n--- results/runs/ (E4 runs) ---")
runs_dir = Path('results/runs')
for d in sorted(runs_dir.iterdir()):
    if d.is_dir() and d.name.startswith('E4'):
        total_size = sum(f.stat().st_size for f in d.iterdir() if f.is_file())
        files = sorted([f.name for f in d.iterdir() if f.is_file()])
        print(f"  {d.name}/ ({total_size/1024:.0f} KB): {files}")

print("\n--- results/ (CSV summaries) ---")
for f in sorted(Path('results').iterdir()):
    if f.is_file() and f.suffix == '.csv':
        print(f"  {f.name}")

print("\n=== HOÀN TẤT NOTEBOOK TV4 ===")